In [97]:
import numpy as np
import joblib
import time
from sklearn.metrics import accuracy_score
import warnings
warnings.filterwarnings('ignore')

print("Libraries loaded. Ready for Cascading Ablation Study.")

Libraries loaded. Ready for Cascading Ablation Study.


In [98]:
# 1. Load Data
X_test_full = np.load('artifacts/X_test_scaled.npy')
y_test = np.load('artifacts/y_test_encoded.npy')
indices = joblib.load('artifacts/selected_indices.joblib')
X_test_reduced = X_test_full[:, indices]

# Create Mixed Stream for Stage 0 testing
clean_samples = X_test_reduced[:100]
anomalies = np.random.uniform(low=10, high=50, size=(20, len(indices)))
mixed_stream = np.vstack((clean_samples, anomalies))

# 2. Load ORIGINAL Baseline Models
stage0_if = joblib.load('models/stage0_if.joblib')
stage1_lr = joblib.load('models/stage1_lr.joblib')
stage1b_svm = joblib.load('models/stage1b_svm.joblib')
stage2_lgbm = joblib.load('models/stage2_lgbm.joblib')

# 3. Load ALTERNATIVE Models
stage0_ocsvm = joblib.load('models/stage0_ocsvm.joblib')
stage1_dt = joblib.load('models/stage1_dt.joblib')
stage1b_sgd = joblib.load('models/stage1b_sgd.joblib')
stage2_xgb = joblib.load('models/stage2_xgb.joblib')

print("Data, Original Models, and Alternative Models successfully loaded.")

Data, Original Models, and Alternative Models successfully loaded.


In [99]:
def evaluate_end_to_end_pipeline(name, m0, m1, m1b, m2, X_clean, y_clean, X_mixed):
    print("==========================================================")
    print(f" END-TO-END PIPELINE: {name}")
    print("==========================================================")

    # --- 1. STAGE 0 PERFORMANCE (Anomaly Detection) ---
    start = time.perf_counter()
    preds_0 = m0.predict(X_mixed)
    t0_ms = ((time.perf_counter() - start) / len(X_mixed)) * 1_000_000
    
    caught = np.sum(preds_0[100:] == -1)
    fps = np.sum(preds_0[:100] == -1)

    # --- 2. CASCADING CLASSIFICATION & LATENCY ---
    # Measure Stage 0 latency on clean data payload
    start = time.perf_counter()
    _ = m0.predict(X_clean)
    t0_clean_ms = ((time.perf_counter() - start) / len(X_clean)) * 1_000_000

    # Measure Stage 1 (Gatekeeper) Routing
    start = time.perf_counter()
    gate_preds = m1.predict(X_clean)
    t1_ms = ((time.perf_counter() - start) / len(X_clean)) * 1_000_000

    static_mask = (gate_preds == 0)
    dynamic_mask = (gate_preds == 1)
    final_preds = np.zeros(len(X_clean), dtype=int)

    # Measure Stage 1B (Static Route)
    t1b_ms = 0
    if np.any(static_mask):
        start = time.perf_counter()
        final_preds[static_mask] = m1b.predict(X_clean[static_mask])
        t1b_ms = ((time.perf_counter() - start) / np.sum(static_mask)) * 1_000_000

    # Measure Stage 2 (Dynamic Route)
    t2_ms = 0
    if np.any(dynamic_mask):
        start = time.perf_counter()
        final_preds[dynamic_mask] = m2.predict(X_clean[dynamic_mask])
        t2_ms = ((time.perf_counter() - start) / np.sum(dynamic_mask)) * 1_000_000

    # --- 3. CALCULATE FINAL COMPREHENSIVE METRICS ---
    end_to_end_acc = accuracy_score(y_clean, final_preds)
    pct_static = np.sum(static_mask) / len(X_clean)
    pct_dynamic = np.sum(dynamic_mask) / len(X_clean)
    
    # Total latency = Stage 0 + Stage 1 + (Weighted Average of Stage 1B and Stage 2)
    total_avg_latency = t0_clean_ms + t1_ms + (pct_static * t1b_ms) + (pct_dynamic * t2_ms)

    print(f"-> Stage 0 Defense  : Caught {caught}/20 Anomalies | {fps}/100 False Positives")
    print(f"-> Data Routing     : {pct_static*100:.1f}% Routed to Fast Model | {pct_dynamic*100:.1f}% Routed to Heavy Model")
    print(f"-> Total Accuracy   : {end_to_end_acc*100:.2f}%")
    print(f"-> Pipeline Latency : {total_avg_latency:.2f} μs per inference\n")

In [109]:
evaluate_end_to_end_pipeline(
    "100% ORIGINAL BASELINE (IF -> LR -> LinearSVM -> LightGBM)", 
    stage0_if, stage1_lr, stage1b_svm, stage2_lgbm, 
    X_test_reduced, y_test, mixed_stream
)

 END-TO-END PIPELINE: 100% ORIGINAL BASELINE (IF -> LR -> LinearSVM -> LightGBM)
-> Stage 0 Defense  : Caught 20/20 Anomalies | 0/100 False Positives
-> Data Routing     : 52.9% Routed to Fast Model | 47.1% Routed to Heavy Model
-> Total Accuracy   : 93.08%
-> Pipeline Latency : 28.26 μs per inference



In [113]:
evaluate_end_to_end_pipeline(
    "SWAP STAGE 0 (SGD One-Class SVM -> LR -> LinearSVM -> LightGBM)", 
    stage0_ocsvm, stage1_lr, stage1b_svm, stage2_lgbm, 
    X_test_reduced, y_test, mixed_stream
)

 END-TO-END PIPELINE: SWAP STAGE 0 (SGD One-Class SVM -> LR -> LinearSVM -> LightGBM)
-> Stage 0 Defense  : Caught 0/20 Anomalies | 0/100 False Positives
-> Data Routing     : 52.9% Routed to Fast Model | 47.1% Routed to Heavy Model
-> Total Accuracy   : 93.08%
-> Pipeline Latency : 9.87 μs per inference



In [117]:
evaluate_end_to_end_pipeline(
    "SWAP STAGE 1 (IF -> Decision Tree -> LinearSVM -> LightGBM)", 
    stage0_if, stage1_dt, stage1b_svm, stage2_lgbm, 
    X_test_reduced, y_test, mixed_stream
)

 END-TO-END PIPELINE: SWAP STAGE 1 (IF -> Decision Tree -> LinearSVM -> LightGBM)
-> Stage 0 Defense  : Caught 20/20 Anomalies | 0/100 False Positives
-> Data Routing     : 52.9% Routed to Fast Model | 47.1% Routed to Heavy Model
-> Total Accuracy   : 93.11%
-> Pipeline Latency : 29.02 μs per inference



In [120]:
evaluate_end_to_end_pipeline(
    "SWAP STAGE 1B (IF -> LR -> SGD Classifier -> LightGBM)", 
    stage0_if, stage1_lr, stage1b_sgd, stage2_lgbm, 
    X_test_reduced, y_test, mixed_stream
)

 END-TO-END PIPELINE: SWAP STAGE 1B (IF -> LR -> SGD Classifier -> LightGBM)
-> Stage 0 Defense  : Caught 20/20 Anomalies | 0/100 False Positives
-> Data Routing     : 52.9% Routed to Fast Model | 47.1% Routed to Heavy Model
-> Total Accuracy   : 92.53%
-> Pipeline Latency : 26.98 μs per inference



In [124]:
evaluate_end_to_end_pipeline(
    "SWAP STAGE 2 (IF -> LR -> LinearSVM -> XGBoost)", 
    stage0_if, stage1_lr, stage1b_svm, stage2_xgb, 
    X_test_reduced, y_test, mixed_stream
)

 END-TO-END PIPELINE: SWAP STAGE 2 (IF -> LR -> LinearSVM -> XGBoost)
-> Stage 0 Defense  : Caught 20/20 Anomalies | 0/100 False Positives
-> Data Routing     : 52.9% Routed to Fast Model | 47.1% Routed to Heavy Model
-> Total Accuracy   : 93.48%
-> Pipeline Latency : 24.93 μs per inference

